[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wdittaya/AI-for-digital-health-course/blob/main/lab05_object_detection_yolo.ipynb)

# Lab 5: การประมวลผลวิดีโอและการตรวจจับวัตถุ (Object Detection) ด้วย YOLO

**รายวิชา:** 3099704 AI for Digital Health (2026) · **คาบเรียน:** วันพุธที่ 14 ตุลาคม 2026 · **เวลาโดยประมาณ:** 90 นาที

## วัตถุประสงค์การเรียนรู้

- ใช้โมเดล YOLO ที่ผ่านการฝึกล่วงหน้า (pretrained) ตรวจจับวัตถุในภาพและวิดีโอ
- ปรับละเอียด (fine-tune) โมเดล YOLO ด้วยชุดข้อมูล (dataset) ทางการแพทย์สาธารณะ เพื่อตรวจหาเนื้องอกสมองในภาพ MRI
- ประเมินผลการตรวจจับด้วย mAP ตรวจพินิจผลการทำนาย และวิเคราะห์ความคลาดเคลื่อนของโมเดล
- เลือกเกณฑ์ความเชื่อมั่น (confidence threshold) และอธิบายการแลกเปลี่ยนระหว่างการตรวจไม่พบรอยโรคกับการแจ้งเตือนผิด
- อธิบายความเชื่อมโยงระหว่างการตรวจจับวัตถุกับกระบวนการทางคลินิก เช่น การตรวจหาติ่งเนื้อ (polyp) ในวิดีโอการส่องกล้องลำไส้ใหญ่

> **⚡ ปฏิบัติการนี้ต้องใช้ GPU** ก่อนเริ่มให้ตั้งค่าใน Colab ดังนี้ `Runtime → Change runtime type → T4 GPU → Save` หาก Colab แจ้งว่าไม่มี GPU ให้ใช้งานในขณะนั้น ผู้เรียนยังสามารถทำปฏิบัติการบน CPU ได้ โดยขั้นตอนการฝึกโมเดลจะใช้จำนวนรอบน้อยลงและใช้เวลานานขึ้น

> **แนวทางการทำปฏิบัติการ** (งานเดี่ยว)
>
> - ✅ **ส่วนหลัก — สำหรับผู้เรียนทุกคน ไม่จำเป็นต้องมีพื้นฐานการเขียนโปรแกรม** ให้รันเซลล์ (cell) ตามลำดับจากบนลงล่าง (คลิกปุ่ม ▶ หรือกด **Shift+Enter**) แบบฝึกหัด ⭐ **Exercise** แต่ละข้อกำหนดให้เปลี่ยนค่าเพียงหนึ่งค่าหรือเติมโค้ดสั้น ๆ หนึ่งบรรทัด และมีเซลล์ **✅ เฉลย** อยู่ถัดไป ผู้เรียนควรทำแบบฝึกหัดด้วยตนเองก่อน หากไม่สามารถดำเนินการต่อได้ ให้รันเซลล์เฉลยแล้วทำขั้นตอนถัดไป จากนั้นตอบคำถาม ✍️ ในเซลล์ข้อความ (ดับเบิลคลิกที่เซลล์ *✍️ คำตอบ* เพื่อพิมพ์) คำตอบเหล่านี้คือชิ้นงานที่ต้องส่ง หากพบปัญหา สามารถสอบถามผู้สอนผ่านช่องแชตของ Zoom
> - 🏆 **โจทย์ท้าทายบนตารางอันดับ — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้** อยู่ท้ายโน้ตบุ๊ก (notebook) และไม่เป็นส่วนหนึ่งของชิ้นงานที่ต้องส่ง

ปฏิบัติการทั้งหมดรันได้บน [Google Colab](https://colab.research.google.com) รุ่นฟรี ส่วนหลักใช้เวลาประมาณ 60–90 นาที (การฝึกโมเดลใช้เวลาประมาณ 3 นาทีบน GPU T4)

In [ ]:
# @title ⚙️ การตั้งค่า — โปรดรันเซลล์นี้ก่อน (NAME และ JOIN_CODE ใช้เฉพาะโจทย์ท้าทายบนตารางอันดับ ซึ่งไม่บังคับ)
!pip -q install gradio_client huggingface_hub

HF_OWNER  = "wdittaya"     # บัญชี Hugging Face ที่ใช้โฮสต์ตารางอันดับของรายวิชา (ผู้สอนเป็นผู้กำหนด)
LAB_ID    = "lab05"
NAME      = ""            # ไม่บังคับ: ชื่อที่จะแสดงบนตารางอันดับแบบสาธารณะ (สามารถใช้นามแฝงได้)
JOIN_CODE = ""            # ไม่บังคับ: รหัสเข้าร่วมซึ่งผู้สอนประกาศในการเรียนการสอนผ่าน Zoom

SPACE_ID  = f"{HF_OWNER}/aidh-leaderboard"
DATA_REPO = f"{HF_OWNER}/aidh-lab-data"

from huggingface_hub import hf_hub_download

def lab_data(path):
    """ดาวน์โหลดไฟล์ข้อมูลทดสอบสาธารณะของรายวิชา 1 ไฟล์ (เช่น 'pima/test_ids.csv') และคืนค่าตำแหน่งไฟล์ในเครื่อง"""
    return hf_hub_download(DATA_REPO, path, repo_type="dataset")

def submit_to_leaderboard(path, lab=LAB_ID):
    """ส่งไฟล์ผลการทำนายไปยังตารางอันดับของรายวิชา และแสดงคะแนนพร้อมอันดับ"""
    from gradio_client import Client, handle_file
    assert NAME and JOIN_CODE, "โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่าก่อน (จำเป็นเฉพาะการส่งผลไปยังตารางอันดับ)"
    client = Client(SPACE_ID, verbose=False)
    print(client.predict(lab=lab, name=NAME, code=JOIN_CODE, file=handle_file(path), api_name="/submit"))

print(f"Leaderboard: https://huggingface.co/spaces/{SPACE_ID}  |  lab={LAB_ID}  name={NAME or '(ยังไม่ได้ระบุ — ตารางอันดับไม่บังคับ)'}")

## เครื่องมือ: Ultralytics YOLO

**การตรวจจับวัตถุ (object detection)** คือการให้โมเดลระบุ *ตำแหน่ง* ของสิ่งที่สนใจในภาพ โดยวาด **กรอบล้อมวัตถุ (bounding box)** พร้อม **ป้ายระบุคลาส (class label)** และ **ค่าความเชื่อมั่น (confidence score)** ระหว่าง 0 ถึง 1 ซึ่งแสดงระดับความมั่นใจของโมเดล **YOLO** ("You Only Look Once") เป็นตระกูลโมเดลตรวจจับวัตถุที่ประมวลผลได้รวดเร็ว แพ็กเกจ `ultralytics` มีโมเดล YOLO ที่ผ่านการฝึกล่วงหน้าและคำสั่งสำหรับฝึกโมเดลที่ใช้งานได้ในบรรทัดเดียว

เซลล์ถัดไปจะติดตั้งแพ็กเกจ (ประมาณ 1 นาที) โหลดโมเดลขนาดเล็ก (`yolo11n` โดย "n" หมายถึง nano ซึ่งเล็กและเร็วที่สุด) และตรวจสอบว่ามี GPU หรือไม่

In [ ]:
!pip -q install ultralytics
from ultralytics import YOLO
import torch, matplotlib.pyplot as plt

print("มี GPU หรือไม่:", torch.cuda.is_available(),
      "→ พร้อมใช้งาน" if torch.cuda.is_available() else "→ โปรดเปิดใช้ GPU: Runtime → Change runtime type → T4 GPU แล้วรันเซลล์อีกครั้ง")
model = YOLO("yolo11n.pt")   # โมเดล nano ที่ฝึกล่วงหน้าด้วยภาพวัตถุทั่วไป (ชุดข้อมูล COCO ดาวน์โหลดอัตโนมัติ)
print("โมเดลรู้จักวัตถุ", len(model.names), "คลาส ตัวอย่างเช่น:", list(model.names.values())[:10])

In [ ]:
# ทดสอบการตรวจจับวัตถุในภาพถ่ายทั่วไป เพื่อตรวจสอบว่าระบบทำงานถูกต้อง
res = model("https://ultralytics.com/images/bus.jpg")
plt.figure(figsize=(8, 6)); plt.imshow(res[0].plot()[..., ::-1]); plt.axis("off"); plt.show()

for box in res[0].boxes:     # แสดงหนึ่งบรรทัดต่อวัตถุที่ตรวจพบ
    print(f"{model.names[int(box.cls)]:>10s}  ความเชื่อมั่น (confidence) = {float(box.conf):.2f}")

💡 **คำอธิบายผลลัพธ์**
โมเดลที่ผ่านการฝึกล่วงหน้าวาดกรอบล้อมบุคคลแต่ละคนและรถประจำทาง พร้อมป้ายระบุคลาสและค่าความเชื่อมั่น โมเดลนี้ไม่เคยได้รับการฝึกด้วยภาพทางการแพทย์ จึงรู้จักเฉพาะวัตถุทั่วไป ในขั้นตอนต่อไปจะ *ปรับละเอียด (fine-tune)* โมเดล กล่าวคือ ฝึกโมเดลเพิ่มเติมสำหรับงานใหม่ด้วยชุดข้อมูลขนาดเล็กที่มีป้ายกำกับ เพื่อให้ตรวจหาเนื้องอกสมองในภาพ MRI ได้

### การประมวลผลวิดีโอ

วิดีโอประกอบด้วยภาพนิ่งจำนวนมากที่เรียงต่อกัน เรียกว่า **เฟรม (frame)** โมเดล YOLO จึงประมวลผลทีละเฟรม ในเซลล์ถัดไปจะประมวลผลคลิปวิดีโอถนนความยาวสั้น โดยเลือกเฉพาะทุกเฟรมที่ 5 (`vid_stride=5` เพื่อลดเวลาประมวลผล) และบันทึกเป็นวิดีโอที่มีกรอบการตรวจจับ หลักการเดียวกันนี้ใช้ในระบบตรวจหา polyp หรือรอยโรคแบบเวลาจริง (real-time) ระหว่างการส่องกล้องหรือการตรวจอัลตราซาวนด์ กล่าวคือ โมเดลเดียวกันประมวลผลทุกเฟรมของภาพวิดีโอที่รับเข้ามา

In [ ]:
# ดาวน์โหลดวิดีโอตัวอย่างและตรวจจับวัตถุทีละเฟรม (ประมาณ 30 วินาทีบน GPU)
!wget -q -O sample.mp4 https://github.com/intel-iot-devkit/sample-videos/raw/master/person-bicycle-car-detection.mp4

frames = []
for i, r in enumerate(model.predict(source="sample.mp4", save=True, vid_stride=5, stream=True, verbose=False)):
    if i % 40 == 0 and len(frames) < 4:          # เก็บเฟรมที่มีกรอบการตรวจจับไว้จำนวนหนึ่งเพื่อแสดงผล
        frames.append(r.plot()[..., ::-1])
    save_dir = r.save_dir
print(f"ประมวลผลแล้ว {i + 1} เฟรม บันทึกวิดีโอที่มีกรอบการตรวจจับไว้ที่: {save_dir}  (ดาวน์โหลดได้จากแผง 📁 Files ด้านซ้าย)")

fig, axes = plt.subplots(1, len(frames), figsize=(4 * len(frames), 3), squeeze=False)
for ax, f in zip(axes[0], frames):
    ax.imshow(f); ax.axis("off")
plt.tight_layout(); plt.show()

💡 **คำอธิบายผลลัพธ์**
โมเดลประมวลผลแต่ละเฟรมที่เลือกไว้แยกจากกัน วาดกรอบการตรวจจับลงบนเฟรม แล้วนำเฟรมมาเรียงต่อกันเป็นไฟล์วิดีโอใหม่ ในระบบวิดีโอทางคลินิก เช่น การส่องกล้องลำไส้ใหญ่ แพทย์จะเห็นกรอบเหล่านี้ปรากฏบนจอภาพแบบเวลาจริงระหว่างการเคลื่อนกล้อง

## การปรับละเอียดโมเดลด้วยข้อมูลทางการแพทย์ (ภาพ MRI เนื้องอกสมอง)

ส่วนนี้ฝึกโมเดลสำหรับงานทางการแพทย์ด้วยชุดข้อมูลสาธารณะ **brain-tumor** ซึ่งมาพร้อมกับ Ultralytics ประกอบด้วยภาพ MRI **ชุดฝึก 893 ภาพ** และ **ชุดตรวจสอบ 223 ภาพ** โดยผู้เชี่ยวชาญวาดกรอบล้อมบริเวณรอยโรคและกำหนดป้ายของแต่ละกรอบเป็น `negative` หรือ `positive` (สองคลาสตามที่ผู้จัดทำชุดข้อมูลกำหนด) ชุดข้อมูลมีขนาดเพียง 4 MB และดาวน์โหลดอัตโนมัติ ขั้นตอนเดียวกันนี้สามารถประยุกต์ใช้กับการตรวจหา polyp ก้อนในปอด (lung nodule) หรือกระดูกหัก

- **ภาพชุดฝึก (training images)** — ภาพที่โมเดลใช้เรียนรู้
- **ภาพชุดตรวจสอบ (validation images)** — ภาพที่แยกไว้ไม่ใช้ฝึก เพื่อวัดประสิทธิภาพของโมเดลกับภาพ MRI ที่โมเดลไม่เคยเห็น
- **Epoch** — การฝึกโมเดลด้วยภาพชุดฝึกครบทุกภาพหนึ่งรอบ จำนวน epoch ที่มากขึ้นทำให้โมเดลเรียนรู้ได้มากขึ้น แต่ใช้เวลานานขึ้น

(หมายเหตุทางเทคนิค: ไฟล์ `brain-tumor.yaml` ระบุรายละเอียดของชุดข้อมูล ป้ายกำกับจัดเก็บในรูปแบบ YOLO `class cx cy w h` ซึ่งปรับสเกลเป็นช่วง 0–1)

> ⏱️ การฝึกโมเดลเป็นขั้นตอนเดียวที่ใช้เวลานาน ประมาณ 10–20 วินาทีต่อ epoch บน GPU T4 และหลายนาทีต่อ epoch บน CPU

In [ ]:
# ⭐ Exercise 1 — แบบฝึกหัดที่ 1: ปรับละเอียดโมเดล YOLO ด้วยภาพ MRI เนื้องอกสมอง
# หน้าที่ของเซลล์: เริ่มจากโมเดล nano ที่ฝึกล่วงหน้า แล้วฝึกเพิ่มเติมด้วยภาพ MRI ชุดฝึก 893 ภาพ
# สิ่งที่ต้องแก้ไข: แทนที่ ... ด้วยจำนวน epoch (จำนวนรอบการฝึก)
# คำแนะนำ: EPOCHS = 10 (ประมาณ 3 นาทีบน GPU T4) หากไม่มี GPU ให้ใช้ EPOCHS = 3 เพื่อลดเวลาการฝึก
EPOCHS = ...

det = YOLO("yolo11n.pt")
det.train(
    data="brain-tumor.yaml",   # มาพร้อมกับ ultralytics และดาวน์โหลดอัตโนมัติ
    epochs=EPOCHS,
    imgsz=416,                 # ปรับขนาดภาพเป็น 416 x 416 พิกเซล
    batch=16,                  # ฝึกครั้งละ 16 ภาพ
)

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
EPOCHS = 10 if torch.cuda.is_available() else 3   # หากใช้ CPU ให้ฝึกเพียงไม่กี่รอบ

det = YOLO("yolo11n.pt")
det.train(
    data="brain-tumor.yaml",   # มาพร้อมกับ ultralytics และดาวน์โหลดอัตโนมัติ
    epochs=EPOCHS,
    imgsz=416,                 # ปรับขนาดภาพเป็น 416 x 416 พิกเซล
    batch=16,                  # ฝึกครั้งละ 16 ภาพ
)

💡 **คำอธิบายผลลัพธ์**
ในแต่ละ epoch โมเดลทำนายกรอบบนภาพ MRI ชุดฝึก เปรียบเทียบกับกรอบของผู้เชี่ยวชาญ และปรับพารามิเตอร์เพียงเล็กน้อยเพื่อลดความคลาดเคลื่อน เมื่อจบแต่ละ epoch ระบบจะวัดผลกับภาพชุดตรวจสอบ กราฟการฝึกด้านล่างแสดงผลดังกล่าว โดยค่า **loss** (ค่าความคลาดเคลื่อนของโมเดล) ควร *ลดลง* และค่า **mAP** (คุณภาพการตรวจจับ ซึ่งอธิบายในแบบฝึกหัดที่ 2) ควร *เพิ่มขึ้น* (ชื่อกราฟเป็นภาษาอังกฤษตามที่ Ultralytics สร้างขึ้น)

In [ ]:
# กราฟการฝึก (Ultralytics บันทึกไว้โดยอัตโนมัติ)
import os
from IPython.display import Image, display
curves = os.path.join(det.trainer.save_dir, "results.png")
display(Image(curves, width=1000)) if os.path.exists(curves) else print("ไม่พบไฟล์ results.png ใน", det.trainer.save_dir)

In [ ]:
# ⭐ Exercise 2 — แบบฝึกหัดที่ 2: ตรวจสอบโมเดลและอ่านค่าตัวชี้วัด
# หน้าที่ของเซลล์: ให้โมเดลที่ฝึกแล้วทำนายภาพ MRI ชุดตรวจสอบ 223 ภาพ และให้คะแนนเทียบกับกรอบของผู้เชี่ยวชาญ
# สิ่งที่ต้องแก้ไข: แทนที่ ... แต่ละตำแหน่งด้วยชื่อตัวชี้วัดตามคำแนะนำ
# คำแนะนำ: ค่า mAP50 รวม คือ  metrics.box.map50   และค่า mAP50-95 คือ  metrics.box.map
metrics = det.val(data="brain-tumor.yaml", imgsz=416)
map50    = ...      # TODO: metrics.box.map50
map50_95 = ...      # TODO: metrics.box.map

print(f"mAP50   : {map50:.3f}")
print(f"mAP50-95: {map50_95:.3f}")
for i, c in enumerate(metrics.box.ap_class_index):          # ค่า AP50 แยกตามคลาส
    print(f"  AP50 ของคลาส '{det.names[int(c)]}': {metrics.box.ap50[i]:.3f}")

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
metrics = det.val(data="brain-tumor.yaml", imgsz=416)
map50    = metrics.box.map50
map50_95 = metrics.box.map

print(f"mAP50   : {map50:.3f}")
print(f"mAP50-95: {map50_95:.3f}")
for i, c in enumerate(metrics.box.ap_class_index):          # ค่า AP50 แยกตามคลาส
    print(f"  AP50 ของคลาส '{det.names[int(c)]}': {metrics.box.ap50[i]:.3f}")

💡 **คำอธิบายผลลัพธ์**
- **IoU** (Intersection over Union) วัดระดับการซ้อนทับระหว่างกรอบที่โมเดลทำนายกับกรอบของผู้เชี่ยวชาญ โดย 1 หมายถึงซ้อนทับกันสมบูรณ์ และ 0 หมายถึงไม่ซ้อนทับกันเลย การทำนายจะนับเป็น *การตรวจพบที่ถูกต้อง* เมื่อ IoU ≥ 0.5 **และ** คลาสถูกต้อง
- **AP50** (Average Precision ที่ IoU 0.5) สรุปประสิทธิภาพของแต่ละคลาส ทั้งด้านความไว (sensitivity / recall) ในการตรวจพบรอยโรค และด้านค่าพยากรณ์บวก (precision / PPV) ในการหลีกเลี่ยงการแจ้งเตือนผิด โดยพิจารณาครอบคลุม *ทุก* ระดับเกณฑ์ความเชื่อมั่น ค่า 1.0 หมายถึงสมบูรณ์
- **mAP50** คือค่าเฉลี่ยของ AP50 ทุกคลาส ส่วน **mAP50-95** มีความเข้มงวดมากกว่า เนื่องจากเฉลี่ยตั้งแต่ IoU 0.5 ถึง 0.95 กรอบจึงต้องมีตำแหน่งที่แม่นยำสูง

เมื่อฝึกเพียงไม่กี่ epoch โมเดล nano มักได้ค่า mAP50 ในระดับปานกลาง ส่วนถัดไปจึงให้ผู้เรียนตรวจพินิจความคลาดเคลื่อนของโมเดลด้วยตนเอง

✍️ **คำถามที่ 1** โปรดระบุว่าคลาสใด (`negative` หรือ `positive`) มีค่า AP50 ต่ำกว่า และจงเสนอสมมติฐานหนึ่งข้อเพื่ออธิบายสาเหตุ (เช่น จำนวนตัวอย่างในชุดฝึกน้อยกว่า รอยโรคมีขนาดเล็กหรือมองเห็นได้ไม่ชัด หรือป้ายกำกับของผู้เชี่ยวชาญไม่สม่ำเสมอ)

✍️ **คำตอบ:** 

## การตรวจพินิจผลการตรวจจับ

ตัวเลขสรุปเพียงค่าเดียว เช่น mAP ไม่ได้แสดง *ลักษณะ* ของความคลาดเคลื่อน ส่วนนี้จึงแสดงภาพ MRI ชุดตรวจสอบที่วาด **ทั้ง** กรอบของผู้เชี่ยวชาญและกรอบของโมเดลบนภาพเดียวกัน

เซลล์เครื่องมือช่วยด้านล่างให้โมเดลทำนายภาพ MRI ชุดตรวจสอบทั้ง 223 ภาพเพียงครั้งเดียว (โดยเก็บแม้แต่กรอบที่มีความเชื่อมั่นต่ำ) และสร้างเครื่องมือ 2 รายการ ได้แก่
- `show_detections(threshold, start=0)` — แสดงภาพ MRI 8 ภาพ โดยแสดงเฉพาะกรอบของโมเดลที่มีความเชื่อมั่น ≥ `threshold`
- `count_errors(thresholds)` — นับจำนวนการตรวจพบที่ถูกต้อง การตรวจไม่พบ และการแจ้งเตือนผิด ตามเกณฑ์ตัดสิน (threshold) หนึ่งค่าหรือหลายค่า

ผู้เรียนไม่จำเป็นต้องอ่านโค้ดในเซลล์นี้ ให้รันเซลล์เท่านั้น

In [ ]:
# @title 🔧 เครื่องมือช่วยตรวจพินิจผลการตรวจจับ — ให้รันเซลล์นี้ { display-mode: "form" }
import numpy as np, pandas as pd
from pathlib import Path
from PIL import Image as PILImage
import matplotlib.patches as patches
from ultralytics.data.utils import check_det_dataset

VAL_DIR = Path(check_det_dataset("brain-tumor.yaml")["val"])      # .../brain-tumor/images/val
LBL_DIR = VAL_DIR.parent.parent / "labels" / VAL_DIR.name         # .../brain-tumor/labels/val
CLASS_NAMES = {int(k): v for k, v in det.names.items()}
VAL_IMAGES = sorted(p for p in VAL_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})


def load_expert_boxes(img_path):
    """กรอบของผู้เชี่ยวชาญในภาพหนึ่งภาพ ในรูป [(cls, x1, y1, x2, y2)] หน่วยพิกเซล"""
    w, h = PILImage.open(img_path).size
    lbl = LBL_DIR / (img_path.stem + ".txt")
    boxes = []
    if lbl.exists():
        for line in lbl.read_text().split("\n"):
            if line.strip():
                c, cx, cy, bw, bh = map(float, line.split()[:5])
                boxes.append((int(c), (cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h))
    return boxes


def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def judge(preds, experts, threshold, iou_thr=0.5):
    """จำแนกกรอบของโมเดลแต่ละกรอบเป็น 'hit' / 'wrong label' / 'false alarm' และระบุว่ากรอบของผู้เชี่ยวชาญแต่ละกรอบตรวจพบหรือไม่"""
    kept = sorted([p for p in preds if p[1] >= threshold], key=lambda p: -p[1])
    found = [False] * len(experts)
    verdicts = []
    for cls, conf, *box in kept:
        best, best_j = 0.0, -1
        for j, (ecls, *ebox) in enumerate(experts):
            if ecls == cls and not found[j] and iou(box, ebox) > best:
                best, best_j = iou(box, ebox), j
        if best >= iou_thr:
            found[best_j] = True
            verdicts.append("hit")
        elif any(iou(box, ebox) >= iou_thr for ecls, *ebox in experts if ecls != cls):
            verdicts.append("wrong label")
        else:
            verdicts.append("false alarm")
    return kept, verdicts, found


# ให้โมเดลทำนายภาพ MRI ชุดตรวจสอบทุกภาพเพียงครั้งเดียว และเก็บทุกกรอบ (conf >= 0.001) ไว้สำหรับกรองภายหลัง
PREDS, EXPERTS = {}, {}
for r in det.predict(source=str(VAL_DIR), conf=0.001, imgsz=416, stream=True, verbose=False):
    name = Path(r.path).name
    PREDS[name] = [(int(c), float(p), *map(float, b))
                   for b, c, p in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist())]
for p in VAL_IMAGES:
    EXPERTS[p.name] = load_expert_boxes(p)
print(f"โมเดลเสนอกรอบที่เป็นไปได้ {sum(len(v) for v in PREDS.values())} กรอบ บนภาพ MRI ชุดตรวจสอบ {len(PREDS)} ภาพ; "
      f"ผู้เชี่ยวชาญวาดกรอบไว้ {sum(len(v) for v in EXPERTS.values())} กรอบ")

COLORS = {"hit": "lime", "wrong label": "orange", "false alarm": "red"}


def show_detections(threshold=0.25, start=0, n=8, only_errors=False):
    """แสดงภาพ MRI ชุดตรวจสอบ n ภาพ พร้อมกรอบของผู้เชี่ยวชาญ (เส้นประ) และกรอบของโมเดลที่มีความเชื่อมั่น >= threshold"""
    # หมายเหตุ: ข้อความบนภาพใช้ภาษาอังกฤษ เนื่องจากฟอนต์เริ่มต้นของ matplotlib บน Colab ไม่รองรับภาษาไทย
    chosen = []
    for p in VAL_IMAGES[start:]:
        kept, verdicts, found = judge(PREDS.get(p.name, []), EXPERTS[p.name], threshold)
        if only_errors and all(v == "hit" for v in verdicts) and all(found):
            continue
        chosen.append((p, kept, verdicts, found))
        if len(chosen) == n:
            break
    if not chosen:
        print("ไม่มีภาพที่จะแสดง (โปรดลดค่า start หรือใช้ only_errors=False)"); return
    cols = 4
    rows = (len(chosen) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4.3 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (p, kept, verdicts, found) in zip(axes.flat, chosen):
        ax.imshow(PILImage.open(p).convert("RGB"))
        for (ecls, x1, y1, x2, y2), ok in zip(EXPERTS[p.name], found):
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ls="--", lw=2,
                                           ec="white" if ok else "yellow"))
            ax.text(x1, y2 + 12, f"expert: {CLASS_NAMES[ecls]}" + ("" if ok else "  ← MISSED"),
                    color="white" if ok else "yellow", fontsize=8, weight="bold")
        for (cls, conf, x1, y1, x2, y2), v in zip(kept, verdicts):
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, lw=2, ec=COLORS[v]))
            ax.text(x1, y1 - 4, f"{CLASS_NAMES[cls]} {conf:.2f} ({v})", color=COLORS[v], fontsize=8, weight="bold")
        n_miss = found.count(False)
        n_fa = sum(v != "hit" for v in verdicts)
        ax.set_title(f"{p.name}\nmissed: {n_miss}  false alarms/wrong label: {n_fa}", fontsize=9)
    fig.suptitle(f"threshold = {threshold}   |   dashed = expert box (yellow = missed)   |   "
                 "solid = model: green hit, orange right place/wrong label, red false alarm", fontsize=10)
    plt.tight_layout(); plt.show()


def count_errors(thresholds=(0.25,)):
    """จำนวนการตรวจพบที่ถูกต้อง การตรวจไม่พบ และการแจ้งเตือนผิด ในภาพ MRI ชุดตรวจสอบทั้ง 223 ภาพ แยกตาม threshold"""
    if isinstance(thresholds, (int, float)):
        thresholds = [thresholds]
    n_expert = sum(len(v) for v in EXPERTS.values())
    rows = []
    for t in thresholds:
        hits = fa = wrong = 0
        for name, experts in EXPERTS.items():
            _, verdicts, _ = judge(PREDS.get(name, []), experts, t)
            hits += verdicts.count("hit"); wrong += verdicts.count("wrong label"); fa += verdicts.count("false alarm")
        n_pred = hits + wrong + fa
        rows.append({"threshold": t, "กรอบของผู้เชี่ยวชาญ (expert boxes)": n_expert, "ตรวจพบถูกต้อง (hits)": hits,
                     "ตรวจไม่พบ (missed)": n_expert - hits, "แจ้งเตือนผิด (false alarms)": fa,
                     "ระบุคลาสผิด (wrong label)": wrong,
                     "ความไว (sensitivity)": round(hits / n_expert, 3),
                     "PPV (precision)": round(hits / n_pred, 3) if n_pred else float("nan")})
    return pd.DataFrame(rows).set_index("threshold")


print("เครื่องมือพร้อมใช้งาน: show_detections(threshold, start=0) และ count_errors([thresholds])")

In [ ]:
# ⭐ Exercise 3 — แบบฝึกหัดที่ 3: ตรวจพินิจผลการตรวจจับของโมเดลที่เกณฑ์ความเชื่อมั่นหนึ่งค่า
# หน้าที่ของเซลล์: แสดงภาพ MRI ชุดตรวจสอบ 8 ภาพ กรอบเส้นประแสดงตำแหน่งรอยโรคที่ผู้เชี่ยวชาญกำหนด
#   กรอบเส้นทึบแสดงกรอบที่โมเดลตรวจพบซึ่งมีความเชื่อมั่น >= THRESHOLD
# สิ่งที่ต้องแก้ไข: กำหนด THRESHOLD เป็นตัวเลขระหว่าง 0 ถึง 1 (โมเดลจะรายงานเฉพาะกรอบที่มีความเชื่อมั่นไม่ต่ำกว่าค่านี้)
#   เมื่อรันแล้ว ให้ทดลองค่าต่ำ (0.1) และค่าสูง (0.7) เพื่อเปรียบเทียบ
#   และเปลี่ยนค่า START (เช่น 8, 16, 40, 100) เพื่อแสดงภาพ MRI ชุดอื่น
# คำแนะนำ: THRESHOLD = 0.25 (ค่าเริ่มต้นของ Ultralytics)
THRESHOLD = ...
START = 0

show_detections(THRESHOLD, start=START)
count_errors(THRESHOLD)

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
THRESHOLD = 0.25
START = 0

show_detections(THRESHOLD, start=START)
count_errors(THRESHOLD)

💡 **คำอธิบายผลลัพธ์** (ข้อความบนภาพเป็นภาษาอังกฤษ)
- **กรอบเส้นประสีขาว** คือรอยโรคที่ผู้เชี่ยวชาญกำหนดและโมเดลตรวจพบ ส่วน **กรอบเส้นประสีเหลือง ← MISSED** คือรอยโรคที่โมเดล *ตรวจไม่พบ* (ผลลบลวง (false negative))
- **กรอบสีเขียว (hit)** คือการตรวจพบที่ถูกต้อง (ผลบวกจริง (true positive)) **กรอบสีแดง (false alarm)** คือการแจ้งเตือนผิด (ผลบวกลวง (false positive)) ซึ่งโมเดลระบุรอยโรคในตำแหน่งที่ผู้เชี่ยวชาญไม่ได้กำหนด **กรอบสีส้ม (wrong label)** คือตำแหน่งถูกต้องแต่ระบุคลาสผิด (สลับระหว่าง `negative` และ `positive`)
- ตัวเลขหลังชื่อคลาสคือค่าความเชื่อมั่นของโมเดล
- ในตาราง **ความไว (sensitivity)** คือสัดส่วนของรอยโรคที่ผู้เชี่ยวชาญกำหนดซึ่งโมเดลตรวจพบ และ **PPV** (ค่าพยากรณ์บวก (positive predictive value / precision)) คือสัดส่วนของกรอบที่โมเดลรายงานซึ่งถูกต้อง

✍️ **คำถามที่ 2** ให้ตรวจพินิจภาพ MRI อย่างน้อย 16 ภาพ (โดยเปลี่ยนค่า `START` หรือรัน `show_detections(THRESHOLD, only_errors=True)` ในเซลล์ใหม่เพื่อแสดงเฉพาะภาพที่มีความคลาดเคลื่อน) จงอธิบาย **รอยโรคที่โมเดลตรวจไม่พบหนึ่งตำแหน่ง** และ **การแจ้งเตือนผิดหรือการระบุคลาสผิดหนึ่งตำแหน่ง** โดยระบุชื่อภาพและลักษณะที่พบ (เช่น ขนาดเล็ก ความเข้มต่ำ หรืออยู่ใกล้ขอบกะโหลกศีรษะ) พร้อมทั้งอธิบายผลกระทบทางคลินิกที่อาจเกิดขึ้นในแต่ละกรณี

✍️ **คำตอบ:** 

In [ ]:
# ⭐ Exercise 4 — แบบฝึกหัดที่ 4: เลือกเกณฑ์ตัดสิน (threshold) สำหรับการใช้งานทางคลินิก
# หน้าที่ของเซลล์: เปรียบเทียบจำนวนการตรวจไม่พบและการแจ้งเตือนผิดที่ threshold หลายค่า บนภาพ MRI ชุดตรวจสอบทั้ง 223 ภาพ
#   แล้วแสดงเฉพาะภาพที่โมเดลมีความคลาดเคลื่อน ณ threshold ที่ผู้เรียนเลือก
# สิ่งที่ต้องแก้ไข: กำหนด MY_THRESHOLD เป็นค่าที่เหมาะสมสำหรับ "เครื่องมือคัดกรอง" ซึ่งส่งภาพ MRI ให้รังสีแพทย์ทบทวน
#   (การตรวจไม่พบเนื้องอกมีผลเสียร้ายแรงกว่าการให้แพทย์ทบทวนภาพเพิ่มเติม)
# คำแนะนำ: พิจารณาจากตาราง และเลือก threshold ที่ให้ความไวสูงโดยมีจำนวนการแจ้งเตือนผิดที่ยอมรับได้ เช่น MY_THRESHOLD = 0.1
display(count_errors([0.05, 0.1, 0.25, 0.5, 0.75]))

MY_THRESHOLD = ...

show_detections(MY_THRESHOLD, only_errors=True)

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
display(count_errors([0.05, 0.1, 0.25, 0.5, 0.75]))

MY_THRESHOLD = 0.1

show_detections(MY_THRESHOLD, only_errors=True)

💡 **คำอธิบายผลลัพธ์**
เมื่อลด threshold โมเดลจะรายงานกรอบมากขึ้น ส่งผลให้ **ตรวจไม่พบรอยโรคน้อยลง** (ความไวสูงขึ้น) แต่ **แจ้งเตือนผิดมากขึ้น** (PPV ลดลง) และเมื่อเพิ่ม threshold จะให้ผลในทิศทางตรงข้าม ตัวโมเดลไม่ได้เปลี่ยนแปลง สิ่งที่เปลี่ยนคือจุดทำงาน (operating point) ที่เลือก ค่า mAP สรุปประสิทธิภาพของโมเดลครอบคลุม *ทุก* threshold แต่ในการใช้งานจริง ทีมผู้ให้บริการทางคลินิกต้องเลือก threshold หนึ่งค่าที่สอดคล้องกับวัตถุประสงค์ของการใช้งาน (การคัดกรองหรือการยืนยันการวินิจฉัย)

✍️ **คำถามที่ 3** โปรดระบุค่า `MY_THRESHOLD` ที่เลือก พร้อมค่าความไวและจำนวนการแจ้งเตือนผิดที่ได้ หากเครื่องมือนี้ออกรายงานผลขั้นสุดท้ายโดยอัตโนมัติ *โดยไม่มี* รังสีแพทย์ตรวจทาน ผู้เรียนพิจารณาว่าควรเลือก threshold ต่างไปจากเดิมหรือไม่ เพราะเหตุใด

✍️ **คำตอบ:** 

✍️ **คำถามที่ 4** สมมติว่านำโมเดลนี้ไปใช้ตรวจหา **polyp ในวิดีโอการส่องกล้องลำไส้ใหญ่ (colonoscopy)** ซึ่งทุกเฟรมที่ตรวจไม่พบอาจหมายถึงรอยโรคที่ถูกมองข้าม จากเนื้อหาในปฏิบัติการนี้ (การประมวลผลวิดีโอทีละเฟรม ความคลาดเคลื่อนที่วิเคราะห์ และการแลกเปลี่ยนในการเลือก threshold) โปรดระบุความเสี่ยงหนึ่งข้อ และมาตรการป้องกันหนึ่งข้อที่ควรกำหนดก่อนนำไปใช้กับผู้ป่วย

✍️ **คำตอบ:** 

## 🏆 โจทย์ท้าทายบนตารางอันดับ (Leaderboard) — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้

> **ส่วนนี้ไม่บังคับ และไม่เป็นส่วนหนึ่งของชิ้นงานที่ต้องส่ง** ผู้เรียนที่ไม่ประสงค์จะทำสามารถข้ามส่วนนี้ได้ โดยชิ้นงานยังคงครบถ้วน

ให้ส่งกรอบที่โมเดลทำนายทั้งหมดบน **ภาพชุด val 223 ภาพ** ในรูปแบบลิสต์ JSON ของ `{image, class, conf, bbox:[x1,y1,x2,y2]}` (พิกัดหน่วยพิกเซล ไฟล์ `lab05_submission.json`) **ตัวชี้วัด: mAP@0.5** คำนวณบนเซิร์ฟเวอร์ (หลักการเดียวกับ `metrics.box.map50`) ควรใช้ **เกณฑ์ความเชื่อมั่นต่ำมาก** (baseline ใช้ `conf=0.001`) เนื่องจากการคำนวณ mAP ต้องใช้รายการกรอบทั้งหมดที่เรียงตามความเชื่อมั่น มิใช่เฉพาะกรอบที่มีความเชื่อมั่นสูง

วิธีส่งผล: ระบุ `NAME` และ `JOIN_CODE` ในเซลล์ ⚙️ การตั้งค่า (ผู้สอนประกาศรหัสเข้าร่วมในการเรียนการสอนผ่าน Zoom) แล้วรันเซลล์ดังกล่าวอีกครั้ง โค้ด baseline ด้านล่างใช้งานได้ทันทีกับโมเดล `det` จากแบบฝึกหัดที่ 1

> **หลักความซื่อสัตย์ทางวิชาการ (honour code):** ป้ายกำกับของภาพชุด val รวมอยู่ในชุดข้อมูลที่ดาวน์โหลดมา ผู้เรียนต้องส่งเฉพาะผลที่โมเดลของตนทำนายเท่านั้น ห้ามนำภาพชุด val ไปใช้ฝึกโมเดล และห้ามแก้ไขกรอบด้วยตนเอง

In [ ]:
# 🏆 Baseline — ส่งออกผลการตรวจจับบนชุด val และส่งผลไปยังตารางอันดับ (ไม่บังคับ)
import json, os
from ultralytics.data.utils import check_det_dataset

def export_detections(model, imgsz=416, path="lab05_submission.json", **predict_kwargs):
    val_dir = check_det_dataset("brain-tumor.yaml")["val"]
    dets = []
    for r in model.predict(source=val_dir, conf=0.001, imgsz=imgsz, stream=True, verbose=False, **predict_kwargs):
        for (x1, y1, x2, y2), c, p in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist()):
            dets.append({"image": os.path.basename(r.path), "class": int(c), "conf": round(p, 4),
                         "bbox": [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)]})
    print(len(dets), "กรอบ บน", len({d["image"] for d in dets}), "ภาพ ->", path)
    with open(path, "w") as f:
        json.dump(dets, f)
    return path

sub = export_detections(det, imgsz=416)   # imgsz ต้องเท่ากับขนาดที่ใช้ฝึก
if NAME and JOIN_CODE:
    submit_to_leaderboard(sub)
else:
    print("โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่า แล้วรันเซลล์นั้นอีกครั้ง เพื่อส่งผลไปยังตารางอันดับ")

**แนวทางการปรับปรุงผลให้สูงกว่า baseline**
- ฝึกนานขึ้น (`epochs=50–100` โดยคำนึงถึงข้อจำกัดด้านเวลาของ Colab รุ่นฟรี) และ/หรือใช้ความละเอียดดั้งเดิมของภาพ `imgsz=512`
- ใช้โมเดลขนาดใหญ่ขึ้น เช่น `yolo11s.pt` หรือ `yolo11m.pt` (หรือตระกูลที่ใหม่กว่า เช่น `yolo26n.pt` หาก `ultralytics` เวอร์ชันที่ใช้รองรับ)
- ปรับการเสริมข้อมูล (data augmentation) เช่น `degrees`, `scale`, `flipud`, `mixup` และอัตราการเรียนรู้ (`lr0`, `cos_lr=True`)
- ใช้การเสริมข้อมูลขณะทดสอบ (test-time augmentation) ในขั้นการทำนาย: `export_detections(model, imgsz=..., augment=True)`
- ใช้ค่า `conf` ต่ำมากในไฟล์ที่ส่งเสมอ เนื่องจาก threshold ที่สูงขึ้นทำให้ค่า mAP ลดลงเท่านั้น

In [ ]:
# 🏆 โมเดลที่ปรับปรุงแล้ว (ไม่บังคับ) — กำหนด IMPROVE = True และปรับการตั้งค่าตามต้องการ
IMPROVE = False

if IMPROVE:
    det2 = YOLO("yolo11s.pt")                       # ทดลองโมเดลขนาดใหญ่ขึ้น
    det2.train(data="brain-tumor.yaml", epochs=50, imgsz=512, batch=16)
    print("mAP50 บนชุด val:", det2.val(data="brain-tumor.yaml", imgsz=512).box.map50)
    sub = export_detections(det2, imgsz=512)
    if NAME and JOIN_CODE:
        submit_to_leaderboard(sub)

## ชิ้นงานที่ต้องส่ง (งานเดี่ยว)

ให้ส่งตามช่องทางที่รายวิชากำหนด ดังนี้
1. โน้ตบุ๊กนี้ที่ **รันครบทุกเซลล์ตามลำดับจากบนลงล่าง** (แสดงกราฟการฝึก ค่า mAP50 / mAP50-95 และภาพ MRI ที่มีกรอบการตรวจจับอย่างน้อยหนึ่งภาพ)
2. คำตอบของ ✍️ **คำถามที่ 1–4**

🏆 *ไม่บังคับ:* ผู้เรียนที่ทำโจทย์ท้าทายบนตารางอันดับ สามารถระบุชื่อที่ใช้บนตารางอันดับและคะแนนสูงสุดไว้ในส่วนนี้ได้

✍️ **ชื่อบนตารางอันดับและคะแนน (ไม่บังคับ):** 